# AI/ML Job Salary Prediction

Predicting salary (USD) for AI/ML roles using job attributes such as experience, education, skills, industry, company, location, remote-work type, and certification.

**Workflow:** Load data → EDA → Preprocessing → Train/Test Split → Model Training (multiple regressors) → Evaluation → Feature Importance → Conclusion

## 1. Load the Dataset
Run the cell below and upload `ai_ml_job_analysis.csv` when prompted (Colab file picker). If the file is already in your Colab working directory, this cell will just skip the upload step.

In [ ]:
import os

# Upload dataset in Google Colab (skips automatically if file already present)
if not os.path.exists('ai_ml_job_analysis.csv'):
    try:
        from google.colab import files
        print("Please upload 'ai_ml_job_analysis.csv'")
        uploaded = files.upload()
    except ImportError:
        pass  # not running in Colab

assert os.path.exists('ai_ml_job_analysis.csv'), "Dataset not found. Please upload ai_ml_job_analysis.csv"
print("Dataset ready ✅")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

df = pd.read_csv('ai_ml_job_analysis.csv')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
print("Shape:", df.shape)
df.info()

In [ ]:
df.describe(include='all').T

In [ ]:
# Missing values
df.isnull().sum()

In [ ]:
# Target variable distribution
plt.figure(figsize=(10,5))
sns.histplot(df['Salary (USD)'], kde=True, bins=30, color='teal')
plt.title('Salary Distribution')
plt.xlabel('Salary (USD)')
plt.show()

In [ ]:
# Numeric feature correlations
numeric_df = df.select_dtypes(include=[np.number]).drop(columns=['ID'])
plt.figure(figsize=(6,5))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap (Numeric Features)')
plt.show()

In [ ]:
# Salary by categorical features
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sns.boxplot(data=df, x='Education', y='Salary (USD)', ax=axes[0,0])
axes[0,0].set_title('Salary by Education')

sns.boxplot(data=df, x='Remote Work', y='Salary (USD)', ax=axes[0,1])
axes[0,1].set_title('Salary by Remote Work Type')

top_titles = df['Job Title'].value_counts().index
sns.boxplot(data=df, x='Job Title', y='Salary (USD)', order=top_titles, ax=axes[1,0])
axes[1,0].set_title('Salary by Job Title')
axes[1,0].tick_params(axis='x', rotation=75)

sns.boxplot(data=df, x='Industry', y='Salary (USD)', ax=axes[1,1])
axes[1,1].set_title('Salary by Industry')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Experience vs Salary
plt.figure(figsize=(9,5))
sns.scatterplot(data=df, x='Experience (Years)', y='Salary (USD)', hue='Education', alpha=0.7)
plt.title('Experience vs Salary')
plt.show()

## 3. Data Preprocessing

Steps:
1. Drop the `ID` column (not predictive).
2. Encode categorical columns:
   - **Ordinal** encoding for `Education` (has a natural order: Bachelors < Masters < PhD).
   - **One-Hot** encoding for `Remote Work` (small, unordered category set).
   - **Frequency/Target-safe** encoding (via `LabelEncoder`) for high-cardinality columns: `Job Title`, `Industry`, `Company`, `Location`, `Certification`.
3. Scale numeric features for the linear model.

In [ ]:
from sklearn.preprocessing import LabelEncoder

data = df.drop(columns=['ID']).copy()

# Ordinal encode Education
education_order = {'Bachelors': 0, 'Masters': 1, 'PhD': 2}
data['Education'] = data['Education'].map(education_order)

# One-hot encode Remote Work
data = pd.get_dummies(data, columns=['Remote Work'], prefix='Remote')

# Label encode high-cardinality categorical columns
label_cols = ['Job Title', 'Industry', 'Company', 'Location', 'Certification']
encoders = {}
for col in label_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))
    encoders[col] = le

data.head()

In [ ]:
X = data.drop(columns=['Salary (USD)'])
y = data['Salary (USD)']

print("Feature columns:", list(X.columns))
print("X shape:", X.shape, "| y shape:", y.shape)

## 4. Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}")

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 5. Model Training

We train and compare several regression models:
- Linear Regression (baseline, uses scaled features)
- Decision Tree Regressor
- Random Forest Regressor
- Gradient Boosting Regressor

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

results = {}

def evaluate(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
    print(f"{name:22s} | MAE: {mae:10,.2f} | RMSE: {rmse:10,.2f} | R2: {r2:.4f}")

# --- Linear Regression (scaled) ---
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
pred_lr = lr.predict(X_test_scaled)
evaluate('Linear Regression', y_test, pred_lr)

# --- Decision Tree ---
dt = DecisionTreeRegressor(max_depth=8, random_state=42)
dt.fit(X_train, y_train)
pred_dt = dt.predict(X_test)
evaluate('Decision Tree', y_test, pred_dt)

# --- Random Forest ---
rf = RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
evaluate('Random Forest', y_test, pred_rf)

# --- Gradient Boosting ---
gb = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42)
gb.fit(X_train, y_train)
pred_gb = gb.predict(X_test)
evaluate('Gradient Boosting', y_test, pred_gb)

## 6. Model Comparison

In [ ]:
results_df = pd.DataFrame(results).T.sort_values('R2', ascending=False)
results_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))

results_df['R2'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('R² Score by Model')
axes[0].set_ylabel('R²')
axes[0].tick_params(axis='x', rotation=30)

results_df['RMSE'].plot(kind='bar', ax=axes[1], color='indianred')
axes[1].set_title('RMSE by Model')
axes[1].set_ylabel('RMSE (USD)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 7. Best Model — Predicted vs Actual

In [ ]:
best_model_name = results_df['R2'].idxmax()
print("Best model:", best_model_name)

preds_map = {
    'Linear Regression': pred_lr,
    'Decision Tree': pred_dt,
    'Random Forest': pred_rf,
    'Gradient Boosting': pred_gb,
}
best_preds = preds_map[best_model_name]

plt.figure(figsize=(7,7))
plt.scatter(y_test, best_preds, alpha=0.6, color='darkorange')
lims = [min(y_test.min(), best_preds.min()), max(y_test.max(), best_preds.max())]
plt.plot(lims, lims, 'k--', label='Perfect Prediction')
plt.xlabel('Actual Salary (USD)')
plt.ylabel('Predicted Salary (USD)')
plt.title(f'Predicted vs Actual Salary — {best_model_name}')
plt.legend()
plt.show()

In [ ]:
# Residual distribution for the best model
residuals = y_test.values - best_preds
plt.figure(figsize=(9,5))
sns.histplot(residuals, kde=True, bins=30, color='purple')
plt.title(f'Residual Distribution — {best_model_name}')
plt.xlabel('Residual (Actual - Predicted)')
plt.show()

## 8. Feature Importance (Random Forest)

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(9,6))
sns.barplot(x=importances.values, y=importances.index, palette='viridis')
plt.title('Feature Importance — Random Forest')
plt.xlabel('Importance')
plt.show()

importances

## 9. Predict Salary for a New Job Profile

Edit the dictionary below to test a custom candidate profile.

In [ ]:
new_profile = {
    'Job Title': 'Machine Learning Engineer',
    'Experience (Years)': 5.0,
    'Education': 'Masters',
    'Skill Count': 12,
    'Industry': 'Technology',
    'Company': 'Google',
    'Location': 'Bengaluru, India',
    'Remote Work': 'Yes',
    'Certification': 'AWS ML Specialty',
}

new_df = pd.DataFrame([new_profile])
new_df['Education'] = new_df['Education'].map(education_order)
new_df = pd.get_dummies(new_df, columns=['Remote Work'], prefix='Remote')

for col in label_cols:
    # unseen categories fall back to the most frequent known label
    known = set(encoders[col].classes_)
    new_df[col] = new_df[col].apply(lambda v: v if v in known else encoders[col].classes_[0])
    new_df[col] = encoders[col].transform(new_df[col].astype(str))

# align columns with training data (add any missing one-hot columns as 0)
for col in X.columns:
    if col not in new_df.columns:
        new_df[col] = 0
new_df = new_df[X.columns]

predicted_salary = rf.predict(new_df)[0]
print(f"Predicted Salary: ${predicted_salary:,.2f}")

## 10. Conclusion

- The dataset contains 500 AI/ML job records with features covering experience, education, skills, industry, company, location, remote-work type, and certifications.
- Tree-based ensemble models (Random Forest / Gradient Boosting) generally capture non-linear relationships between experience, skill count, and salary better than plain Linear Regression.
- `Experience (Years)` and `Skill Count` tend to be the strongest numeric predictors; categorical factors like `Job Title`, `Industry`, and `Location` also contribute meaningfully.
- Next steps you could explore: hyperparameter tuning (`GridSearchCV`), trying `XGBoost`/`LightGBM`, or building a Streamlit app around the trained model for interactive salary predictions.